# Iraqi Voice Studio — تدريب RVC v2 على Google Colab

دفتر Colab الرسمي لمنصة **استوديو الصوت العراقي**.

يُنفّذ عبر سكربت التثبيت الرسمي `scripts/colab/install_colab.sh` (يتعامل تلقائيًا مع GPU/CUDA/uv/Drive/الاستكمال).

المتغيرات القابلة للتعديل في الخلية الأولى:
- `IRAQI_VOICE_REPO`: مستودع git للمشروع (انسخه إلى Drive أو GitHub خاص بك أولاً).
- `DRIVE_DIR`: مجلد Drive الذي تُحفظ فيه النماذج ونقاط التحقق (افتراضي `RVC_Iraqi`).
- ضع ملفات البيانات الصوتية العراقية (10–60 دقيقة، نظيفة) في `Drive/MyDrive/<DRIVE_DIR>/datasets/<اسم>/`.

In [ ]:
# @title 1) الإعداد والتحقق من البيئة (GPU / CUDA / Drive)
IRAQI_VOICE_REPO = ""  # @param {type:"string"}
DRIVE_DIR = "RVC_Iraqi"  # @param {type:"string"}

import os
os.environ["IRAQI_VOICE_REPO"] = IRAQI_VOICE_REPO
os.environ["DRIVE_DIR"] = DRIVE_DIR

!nvidia-smi -L || echo "NO GPU"
!python3 --version
print("تبقى الخليتان التاليتان: التثبيت ثم التدريب.")

In [ ]:
# @title 2) التثبيت الرسمي (uv + بيئة + أصول مُسبقة + ربط Drive)
# إن كنت قد نسخت المستودع إلى Drive فاستخدم مساره بدلاً من IRAQI_VOICE_REPO.
import os
if os.environ.get("IRAQI_VOICE_REPO"):
    !rm -rf /content/iraqi-voice-platform
    !git clone --depth 1 "$IRAQI_VOICE_REPO" /content/iraqi-voice-platform
    %cd /content/iraqi-voice-platform

!bash scripts/colab/install_colab.sh
print("\n>>> التثبيت اكتمل. جهّز بياناتك في Drive ثم شغّل الخلية التالية. <<<")

In [ ]:
# @title 3) تجهيز البيانات العراقية (تحقق → قص → F0 → ميزات → فهرس)
EXPERIMENT = "iraqi_voice"  # @param {type:"string"}
DATASET_IN_DRIVE = "RVC_Iraqi/datasets/my_voice"  # @param {type:"string"}
SR = "40k"  # @param ["32k", "40k", "48k"]
F0_METHOD = "rmvpe"  # @param ["rmvpe", "fcpe", "pm"]

!source .colab_env 2>/dev/null || true
!uv run rvc-platform data-check "/content/drive/MyDrive/$DATASET_IN_DRIVE"
!uv run rvc-platform prepare-dataset "/content/drive/MyDrive/$DATASET_IN_DRIVE" "$EXPERIMENT" --sr "$SR" --f0-method "$F0_METHOD"

In [ ]:
# @title 4) التدريب (يُستكمل تلقائيًا من آخر Checkpoint عند إعادة التشغيل)
EPOCHS = 200  # @param {type:"integer"}
BATCH = 6  # @param {type:"integer"}

!source .colab_env 2>/dev/null || true
!uv run rvc-platform train start --experiment "$EXPERIMENT" --epochs $EPOCHS --batch $BATCH --sr "$SR"

In [ ]:
# @title 5) (اختياري) واجهة الويب عبر نفق Cloudflare مؤقت
# ملاحظة: النفق يُفعّل المصادقة تلقائيًا — الرمز في logs/auth.token
!source .colab_env 2>/dev/null || true
!uv run rvc-platform web --host 0.0.0.0 --tunnel